[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/typer-certified/notebooks/day-01-typer-basics-commands-arguments.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Typer Basics — Commands, Arguments & Help
**certified-journeys / typer-certified** &nbsp;|&nbsp; Foundation

> **Goal for today:** Install Typer, understand how Python type hints become a CLI spec, and build a multi-command app in 30 minutes.

---
## Why Typer?

Typer turns Python type hints directly into a CLI spec. Your function signature *is* your interface — no argparse boilerplate, no manual help strings.

| Concept | What it means |
|---|---|
| `def cmd(name: str)` | `name` is a required positional argument |
| `def cmd(count: int = 1)` | `--count` is optional with default 1; Typer enforces int |
| `@app.command()` | Registers the function as a runnable CLI command |
| Docstring | Becomes the `--help` description — zero extra code |


In [ ]:
%pip install -q 'typer[all]'


---
## Step 1 · Your first Typer command

The simplest Typer app is one function decorated with `@app.command()`. Arguments become positional; parameters with defaults become `--options`.


In [ ]:
import typer
from typer.testing import CliRunner

app = typer.Typer()
runner = CliRunner()

@app.command()
def greet(name: str, count: int = 1):
    """Greet NAME a number of times."""
    for _ in range(count):
        typer.echo(f"Hello, {name}!")

# Invoke via CliRunner (no subprocess needed)
result = runner.invoke(app, ["Alice", "--count", "3"])
print(result.output)

# --help is generated for free
help_result = runner.invoke(app, ["--help"])
print(help_result.output)


**What just happened?**
- **`name: str`** → required positional argument; Typer errors if omitted
- **`count: int = 1`** → `--count` flag with default 1; Typer auto-converts `"3"` → `int`
- The docstring became the `--help` description — no extra code at all
- **`CliRunner`** lets you test the full CLI in-process without spawning a subprocess


---
## Step 2 · Multi-command apps

Real CLIs have subcommands (`git commit`, `git push`). With Typer, decorate multiple functions on the same `app`:

- Each `@app.command()` → one subcommand, listed automatically in `--help`
- `typer.Exit(code=1)` → clean error exit signalling failure to the shell
- `typer.echo(..., err=True)` → writes to stderr, keeping stdout clean for piping


In [ ]:
files_app = typer.Typer(name='files', help='File inspection tools.')

@files_app.command()
def count(filepath: str):
    """Count lines in FILEPATH."""
    try:
        with open(filepath) as f:
            typer.echo(f"{filepath}: {sum(1 for _ in f)} lines")
    except FileNotFoundError:
        typer.echo(f"Error: {filepath} not found", err=True)
        raise typer.Exit(code=1)

@files_app.command()
def info(filepath: str):
    """Show size and existence of FILEPATH."""
    import os
    if os.path.exists(filepath):
        typer.echo(f"{filepath}: {os.path.getsize(filepath):,} bytes")
    else:
        typer.echo(f"Not found: {filepath}", err=True)
        raise typer.Exit(code=1)

@files_app.command()
def head(filepath: str, n: int = 5):
    """Show first N lines of FILEPATH."""
    try:
        with open(filepath) as f:
            for i, line in enumerate(f):
                if i >= n: break
                typer.echo(line, nl=False)
    except FileNotFoundError:
        typer.echo(f"Not found: {filepath}", err=True); raise typer.Exit(1)

# Check --help shows all 3 commands
r = runner.invoke(files_app, ["--help"])
print(r.output)

# Error path: non-existent file
r2 = runner.invoke(files_app, ["count", "missing.txt"])
print(f"exit code: {r2.exit_code}")


**What just happened?**
- Three `@files_app.command()` decorators → three subcommands, all auto-listed in `--help`
- **`typer.Exit(code=1)`** raises a clean exit; non-zero tells the shell something went wrong
- `typer.echo(..., err=True)` writes to stderr — stdout stays clean for piping to other tools
- Missing file → exit code 1; Typer does **not** print a Python traceback to the user


---
## Step 3 · typer.Argument() and rich help markup

`typer.Argument()` gives you explicit control over positional args:

| Syntax | Effect |
|---|---|
| `typer.Argument(...)` | `...` (Ellipsis) = required, no default |
| `typer.Argument('world')` | Optional with string default |
| `help='...'` | Shown in `--help` output |

With `rich_markup_mode='rich'`, docstrings and help strings accept Rich markup tags.


In [ ]:
rich_app = typer.Typer(rich_markup_mode='rich')

@rich_app.command(epilog='See [link=https://typer.tiangolo.com]typer.tiangolo.com[/link]')
def process(
    filename: str = typer.Argument(..., help='[bold]Input[/bold] file to process'),
    verbose:  bool = typer.Option(False, '--verbose', '-v', help='Enable verbose output'),
):
    """
    [bold green]Process[/bold green] a file.

    [dim]Supports CSV, JSON, and plain text.[/dim]
    """
    if verbose:
        typer.echo(f'Processing: {filename}')
    typer.echo(f'Done: {filename}')

r = runner.invoke(rich_app, ['--help'])
print(r.output)


**What just happened?**
- **`typer.Argument(...)`** — the `...` (Ellipsis) marks it required; Typer errors cleanly if absent
- `rich_markup_mode='rich'` enables Rich markup in docstrings and help strings
- The `epilog=` appears as a footer in `--help` — good place for links or usage examples
- Short flag `-v` is declared alongside `--verbose` in one `typer.Option()` call


In [ ]:
# Challenge: Build a `notes` CLI with 3 subcommands:
#   add TEXT   → prints 'Saved: {text}'
#   list       → prints 'Listing notes...'
#   delete ID  → prints 'Deleted note {id}' (ID is int)
# Requirements: typed args, docstrings, all 3 invocations exit code 0

notes_app = typer.Typer(help='A simple notes CLI.')
# Your solution here


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `@app.command()` | Registers function as CLI command; docstring → `--help` |
| `arg: str` | Required positional argument |
| `opt: int = N` | Optional `--opt` flag with default N; Typer enforces type |
| `typer.Exit(code=1)` | Clean error exit — non-zero signals failure to the shell |
| `CliRunner.invoke()` | Test full CLI in-process; check `.exit_code` and `.output` |

> **Tip:** Typer leverages Python type hints directly — your function signature IS your CLI spec. Write clear types and you get argument validation and `--help` for free.

---
## What's next

**Day 2** → `typer.Option()` deep dive, built-in types (`Choice`, `Path`), `--version` callback with `is_eager=True`, and testing with `CliRunner`.

Mark Day 1 complete in your [tracker](../index.html).
